<a href="https://colab.research.google.com/github/ChaMooKwan/SunMoon-Univ.-Machin-Learning-Project/blob/main/%EA%B8%B0%EA%B3%84%ED%95%99%EC%8A%B5%ED%94%84%EB%A1%9C%EC%A0%9D%ED%8A%B8_ko_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 모델 학습 코드

In [ ]:
!pip install -q transformers torch sentencepiece

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer

# 1. 토크나이저와 기본 KoBERT 모델 로드 (trust_remote_code=True 필수)
model_name = "skt/kobert-base-v1"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
bert_base = AutoModel.from_pretrained(model_name)

# 2. 멀티 태스크(ABSA)를 위한 커스텀 모델 클래스 정의
class ABSAKoBERT(nn.Module):
    def __init__(self, bert_model):
        super(ABSAKoBERT, self).__init__()
        self.bert = bert_model

        # BERT의 출력 벡터 크기 (기본 768)
        hidden_size = self.bert.config.hidden_size

        # 첫 번째 머리 (Head): 속성 존재 여부 (0 or 1 -> 2개 클래스)
        self.confidence_classifier = nn.Linear(hidden_size, 2)

        # 두 번째 머리 (Head): 감성 극성 (부정0, 중립1, 긍정2 -> 3개 클래스)
        self.polarity_classifier = nn.Linear(hidden_size, 3)

    def forward(self, input_ids, attention_mask):
        # 1) 입력값을 BERT에 통과시킴
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        # 2) [CLS] 토큰의 벡터만 추출 (인덱스 0번. 문장 전체의 의미를 담고 있음)
        cls_token_output = outputs.last_hidden_state[:, 0, :]

        # 3) 추출한 벡터를 각각의 분류기로 보냄
        confidence_logits = self.confidence_classifier(cls_token_output)
        polarity_logits = self.polarity_classifier(cls_token_output)

        return confidence_logits, polarity_logits

# 3. 모델 객체 생성 및 GPU(또는 CPU) 할당
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ABSAKoBERT(bert_base).to(device)

print("모델 로드 및 구조 설정 완료! 현재 사용 기기:", device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/535 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/371k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

모델 로드 및 구조 설정 완료! 현재 사용 기기: cuda


In [ ]:
import torch
from torch.utils.data import Dataset

class ABSADataset(Dataset):
    def __init__(self, texts, aspects, confidences, polarities, tokenizer, max_len=128):
        self.texts = texts
        self.aspects = aspects
        self.confidences = confidences
        self.polarities = polarities
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        aspect = str(self.aspects[item])

        # 1. 텍스트와 속성을 함께 토큰화 (예: [CLS] 문장 [SEP] 속성 [SEP])
        encoding = self.tokenizer(
            text,
            aspect,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )

        # 2. 감성 라벨 변환 (-1, 0, 1 -> 0, 1, 2)
        # PyTorch의 CrossEntropyLoss는 음수 인덱스를 받을 수 없기 때문입니다.
        # 이 부분은 이미 데이터프레임에서 처리되었으므로 추가적인 +1은 필요 없습니다.
        polarity_label = self.polarities[item]

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            # 커스텀 모델에 전달할 두 개의 정답
            'confidence_labels': torch.tensor(self.confidences[item], dtype=torch.long),
            'polarity_labels': torch.tensor(polarity_label, dtype=torch.long)
        }

#### 데이터 세팅

In [ ]:
#데이터 불러오기
import pandas as pd

train_data = pd.read_csv('Training_비율통합.csv')
valid_data = pd.read_csv('Validation_비율통합.csv')

train_data['AspectConfidence'] = 1
valid_data['AspectConfidence'] = 1

print(train_data['Aspect'].value_counts(),train_data.shape)
print(valid_data['Aspect'].value_counts(),valid_data.shape)

train_data = train_data[~train_data['Aspect'].str.contains('소비전력')]
valid_data = valid_data[~valid_data['Aspect'].str.contains('소비전력')]

Aspect
가격         10852
기능          9368
편의성         5881
조작성         5866
사이즈         5439
품질          5382
디자인         4606
음량/음질       4598
화질          4345
색상          3966
제품구성        3408
무게          3029
제조일/제조사     2849
시간/속도       2749
배터리         1557
소음          1390
내구성         1237
용량           894
소재           667
소비전력          38
Name: count, dtype: int64 (78121, 24)
Aspect
가격         2713
기능         2342
편의성        1471
조작성        1466
사이즈        1359
품질         1346
디자인        1151
음량/음질      1149
화질         1087
색상          992
제품구성        852
무게          757
제조일/제조사     712
시간/속도       687
배터리         389
소음          348
내구성         310
용량          223
소재          168
소비전력          9
Name: count, dtype: int64 (19531, 24)


In [ ]:
print(train_data['Aspect'].unique())
print(train_data['Aspect'].unique())
print(train_data.shape)
print(valid_data.shape)

['가격' '시간/속도' '품질' '기능' '조작성' '디자인' '용량' '화질' '편의성' '소재' '소음' '음량/음질'
 '사이즈' '무게' '제조일/제조사' '색상' '배터리' '제품구성' '내구성']
['가격' '시간/속도' '품질' '기능' '조작성' '디자인' '용량' '화질' '편의성' '소재' '소음' '음량/음질'
 '사이즈' '무게' '제조일/제조사' '색상' '배터리' '제품구성' '내구성']
(78083, 24)
(19522, 24)


In [ ]:
#데이터셋 생성
train_data = train_data[['SentimentText', 'Aspect', 'SentimentPolarity', 'AspectConfidence']]
train_data['SentimentPolarity'] = train_data['SentimentPolarity'] + 1

def create_negative_aspect_samples(data):
    negative_data = []

    # 각 aspect의 positive 개수
    aspect_counts = train_data['Aspect'].value_counts()

    for target_aspect, count in aspect_counts.items():

        # target_aspect가 아닌 문장들만 후보
        candidate_rows = data[
            data['Aspect'] != target_aspect
        ]

        # 중복 허용 여부
        sampled_rows = candidate_rows.sample(
            n=count,
            replace=len(candidate_rows) < count,
            random_state=42
        )

        for _, row in sampled_rows.iterrows():

            negative_data.append({
                "SentimentText": row["SentimentText"],
                "Aspect": target_aspect,
                "SentimentPolarity": -1,   # mask
                "AspectConfidence": 0
            })

    return negative_data

negative_train_data = create_negative_aspect_samples(train_data)

#기존 데이터 셋과 합치기
train_data = pd.concat([train_data, pd.DataFrame(negative_train_data)], ignore_index=True)

train_data.describe()

,SentimentPolarity,AspectConfidence
count,156166.000000,156166.000000
mean,0.262304,0.500000
std,1.392867,0.500002
min,-1.000000,0.000000
25%,-1.000000,0.000000
50%,-0.500000,0.500000
75%,2.000000,1.000000
max,2.000000,1.000000


In [ ]:
valid_data.describe()

,SentimentPolarity,가격,기능,내구성,디자인,무게,배터리,사이즈,색상,소비전력,...,시간/속도,용량,음량/음질,제조일/제조사,제품구성,조작성,편의성,품질,화질,AspectConfidence
count,19522.000000,19522.000000,19522.000000,19522.000000,19522.000000,19522.000000,19522.000000,19522.000000,19522.000000,19522.0,...,19522.000000,19522.000000,19522.000000,19522.000000,19522.000000,19522.000000,19522.000000,19522.000000,19522.000000,19522.0
mean,0.524588,0.138971,0.119967,0.015880,0.058959,0.038777,0.019926,0.069614,0.050814,0.0,...,0.035191,0.011423,0.058857,0.036472,0.043643,0.075095,0.075351,0.068948,0.055681,1.0
std,0.832671,0.345926,0.324932,0.125013,0.235554,0.193067,0.139750,0.254501,0.219624,0.0,...,0.184267,0.106269,0.235362,0.187465,0.204305,0.263551,0.263963,0.253372,0.229310,0.0
min,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
50%,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
75%,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0


In [ ]:
aspect_sentiment_ratio = pd.crosstab(
    train_data['Aspect'],
    train_data['SentimentPolarity'],
    normalize='index'
)
print(aspect_sentiment_ratio)

SentimentPolarity   -1         0         1         2
Aspect                                              
가격                 0.5  0.046812  0.011519  0.441670
기능                 0.5  0.069012  0.014197  0.416791
내구성                0.5  0.295473  0.022635  0.181892
디자인                0.5  0.064155  0.010204  0.425640
무게                 0.5  0.170518  0.025916  0.303566
배터리                0.5  0.237315  0.011882  0.250803
사이즈                0.5  0.104983  0.016179  0.378838
색상                 0.5  0.097705  0.010590  0.391704
소음                 0.5  0.189568  0.025180  0.285252
소재                 0.5  0.310345  0.013493  0.176162
시간/속도              0.5  0.126046  0.019825  0.354129
용량                 0.5  0.179530  0.028523  0.291946
음량/음질              0.5  0.103415  0.031100  0.365485
제조일/제조사            0.5  0.089856  0.005090  0.405054
제품구성               0.5  0.180898  0.015258  0.303844
조작성                0.5  0.141408  0.019945  0.338646
편의성                0.5  0.110100  0.009522  0.

In [ ]:
texts = train_data['SentimentText'].tolist()
aspects = train_data['Aspect'].tolist()
confidence = train_data['AspectConfidence'].tolist()
polarities = train_data['SentimentPolarity'].tolist()

test_texts = valid_data['SentimentText'].tolist()
test_aspects = valid_data['Aspect'].tolist()
test_confidence = valid_data['AspectConfidence'].tolist()
test_polarities = valid_data['SentimentPolarity'].tolist()

# 데이터셋 생성
test_dataset = ABSADataset(test_texts, test_aspects, test_confidence, test_polarities, tokenizer)
print("테스트 데이터셋 준비 완료! 첫 번째 데이터 샘플:\n", test_dataset[0])
train_dataset = ABSADataset(texts, aspects, confidence, polarities, tokenizer)
print("데이터셋 준비 완료! 첫 번째 데이터 샘플:\n", train_dataset[0])

테스트 데이터셋 준비 완료! 첫 번째 데이터 샘플:
 {'input_ids': tensor([  1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
          1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
          1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
          1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
          1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
          1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
          1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
          1, 517, 493,   0, 517, 492,   0, 517,   0, 490, 494,   0, 517,   0,
        493,   0, 490,   0, 517,   0, 491,   0,   3, 517,   0, 491, 494,   0,
          3,   2]), 'attention_mask': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [ ]:
from transformers import Trainer, TrainingArguments
import torch.nn as nn

class ABSATrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # 1. 입력 데이터 추출
        input_ids = inputs.get("input_ids")
        attention_mask = inputs.get("attention_mask")
        confidence_labels = inputs.get("confidence_labels")
        polarity_labels = inputs.get("polarity_labels")

        # 2. 모델 예측값 뽑기 (우리가 만든 ABSAKoBERT의 forward 결과)
        confidence_logits, polarity_logits = model(input_ids, attention_mask)

        # 3. 각각의 손실(Loss) 계산 함수 선언
        # polarity_labels에 -1이 있을 수 있으므로 ignore_index=-1을 추가
        loss_fct_confidence = nn.CrossEntropyLoss(reduction='none')
        loss_fct_polarity = nn.CrossEntropyLoss(reduction='none', ignore_index=-1)

        # 4. 오차 계산
        loss_confidence = loss_fct_confidence(confidence_logits, confidence_labels).mean()

        # 감성 오차를 구한 뒤
        loss_polarity = loss_fct_polarity(polarity_logits, polarity_labels)

        # 💡 핵심 로직: 정답(confidence_labels)이 1인 경우만 감성 오차를 살리고, 0이면 곱해서 오차를 0으로 날려버림!
        # ignore_index=-1 설정으로 인해 -1 라벨은 이미 계산에서 제외되지만, 이 로직은 confidence_labels=0일 때도 polarity loss를 무시하도록 합니다.
        loss_polarity = (loss_polarity * confidence_labels).mean()

        # 5. 최종 오차 = 두 오차의 합
        total_loss = loss_confidence + loss_polarity

        return (total_loss, (confidence_logits, polarity_logits)) if return_outputs else total_loss

print("커스텀 Trainer 클래스 정의 완료!")

커스텀 Trainer 클래스 정의 완료!


#### 아래의 코드 블럭에서 나오는 report 결과 무시 후 다음 블럭 참조

In [ ]:
# 학습 환경 설정
training_args = TrainingArguments(
    output_dir='./results',          # 모델 가중치가 저장될 폴더
    num_train_epochs=3,              # 전체 데이터 학습 횟수
    per_device_train_batch_size=128,  # 한 번에 학습할 데이터 개수 (코랩 GPU 메모리에 맞춰 조절)
    logging_steps=10,                # 10스텝마다 로그 출력
    save_strategy="epoch",           # 에포크마다 모델 저장
    learning_rate=3e-5,              # BERT 미세조정에 적합한 학습률

    # 💡 핵심 추가 옵션: 모델 forward에 없는 이름의 데이터라도 절대 지우지 마!
    remove_unused_columns=False,
)

# 우리가 개조한 ABSATrainer에 조립
trainer = ABSATrainer(
    model=model,                     # 이전 스텝에서 만든 ABSAKoBERT 객체
    args=training_args,
    train_dataset=train_dataset,     # 1단계에서 만든 데이터셋
)

# 학습 시작!
trainer.train()

# 해당 경로에 폴더가 없으면 새로 만듭니다.
os.makedirs(drive_save_path, exist_ok=True)

# 3. 모델 가중치 및 토크나이저 저장
torch.save(model.state_dict(), f"{drive_save_path}/model_weights2.pth")
tokenizer.save_pretrained(drive_save_path)

print(f"구글 드라이브 경로({drive_save_path})에 모델과 토크나이저가 안전하게 저장되었습니다!")

# 학습 Report (성능 평가) 출력하기
import torch
import numpy as np
from sklearn.metrics import classification_report
from torch.utils.data import DataLoader

# 1. 평가 모드 전환 (Dropout 등을 끔)
model.eval()

# 2. 테스트 데이터를 배치 단위로 불러오기 위한 DataLoader 설정 (순서 유지)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

true_conf, pred_conf = [], []
true_pol, pred_pol = [], []
aspect_for_pol = []  # 감성 극성 정답/예측이 어느 Aspect에 해당하는지 저장하기 위한 리스트

# 3. 모델에 테스트 데이터를 넣고 결과 뽑기
with torch.no_grad():   # 평가할 때는 기울기(Gradient) 계산을 하지 않아 메모리를 절약합니다.
    for batch_idx, batch in enumerate(test_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        # 모델 예측
        conf_logits, pol_logits = model(input_ids, attention_mask)

        # 가장 높은 확률의 인덱스를 정답으로 채택
        conf_preds = torch.argmax(conf_logits, dim=1).cpu().numpy()
        pol_preds = torch.argmax(pol_logits, dim=1).cpu().numpy()

        # 실제 정답과 예측값 저장
        pred_conf.extend(conf_preds)
        true_conf.extend(batch['confidence_labels'].numpy())

        # 속성이 존재(1)하는 경우에 대해서만 감성 정답/예측값을 수집합니다.
        for i, conf_val in enumerate(batch['confidence_labels'].numpy()):
            if conf_val == 1:
                pred_pol.append(pol_preds[i])
                true_pol.append(batch['polarity_labels'][i].numpy())

                # 전체 데이터셋 기준의 인덱스를 계산하여 해당 Aspect를 함께 저장
                global_idx = batch_idx * test_loader.batch_size + i
                aspect_for_pol.append(test_aspects[global_idx])

# 슬라이싱 및 필터링을 쉽게 하기 위해 리스트를 Numpy 배열로 변환
true_conf = np.array(true_conf)
pred_conf = np.array(pred_conf)
true_pol = np.array(true_pol)
pred_pol = np.array(pred_pol)

test_aspects_arr = np.array(test_aspects)  # 전체 테스트 데이터의 Aspect 배열
aspect_pol_arr = np.array(aspect_for_pol)  # 속성이 1인 데이터들의 Aspect 배열

# 4. Aspect Unique 별로 리포트 출력
unique_aspects = np.unique(test_aspects_arr)

print("============ Aspect 별 성능 평가 리포트 ============\n")

for aspect in unique_aspects:
    print(f"[{aspect}] Aspect 리포트")
    print("-" * 50)

    # 1) 속성 존재 여부 (Aspect Confidence) 분리 및 출력
    # idx_conf = (test_aspects_arr == aspect)
    # if np.sum(idx_conf) > 0:
    #     print(f"• {aspect} - 속성 존재 여부 (Aspect Confidence)")
    #     print(classification_report(
    #         true_conf[idx_conf],
    #         pred_conf[idx_conf],
    #         labels=[0, 1], # 라벨 고정 (경고 방지)
    #         target_names=["없음(0)", "있음(1)"],
    #         zero_division=0
    #     ))

    # 2) 감성 극성 (Sentiment Polarity) 분리 및 출력
    idx_pol = (aspect_pol_arr == aspect)
    if np.sum(idx_pol) > 0:
        print(f"• {aspect} - 감성 극성 (Sentiment Polarity)")
        print(classification_report(
            true_pol[idx_pol],
            pred_pol[idx_pol],
            labels=[0, 1, 2], # 라벨 고정 (부정/중립/긍정 중 없는 클래스가 있어도 에러 안 나게 함)
            target_names=["부정(-1)", "중립(0)", "긍정(1)"],
            zero_division=0
        ))

    print("=" * 50, "\n")

Step,Training Loss
10,1.030933
20,1.048225
30,1.007563
40,1.021225
50,1.047973
60,1.021048
70,1.006778
80,0.961328
90,0.984424
100,0.998526


구글 드라이브 경로(/content/drive/MyDrive/my_kobert_absa)에 모델과 토크나이저가 안전하게 저장되었습니다!
============ Aspect 별 성능 평가 리포트 ============

[가격] Aspect 리포트
--------------------------------------------------
• 가격 - 감성 극성 (Sentiment Polarity)
              precision    recall  f1-score   support

      부정(-1)       0.06      0.22      0.09        63
       중립(0)       0.50      0.00      0.01      2396
       긍정(1)       0.00      0.00      0.00         0

   micro avg       0.01      0.01      0.01      2459
   macro avg       0.19      0.08      0.03      2459
weighted avg       0.49      0.01      0.01      2459


[기능] Aspect 리포트
--------------------------------------------------
• 기능 - 감성 극성 (Sentiment Polarity)
              precision    recall  f1-score   support

      부정(-1)       0.07      0.28      0.12        67
       중립(0)       0.40      0.00      0.01      1952
       긍정(1)       0.00      0.00      0.00         0

   micro avg       0.01      0.01      0.01      2019
   macro avg       0.1

## 학습된 모델 업로드 후 검증 및 평가

In [ ]:
# 1. 'my_kobert_absa'폴더를 내 드라이브에 업로드
# 2. 'Validation_비율통합.csv'를 RAM에 업로드
# 3. 코드 실행

In [ ]:
from google.colab import drive
import os
import torch

# 1. 구글 드라이브 마운트 (연결)
drive.mount('/content/drive')

# 2. 내 구글 드라이브 안에 저장할 폴더 경로 지정
# '/content/drive/MyDrive/' 까지가 내 드라이브의 최상단(루트) 경로입니다.
drive_save_path = '/content/drive/MyDrive/my_kobert_absa'

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
from google.colab import drive

# 1. 구글 드라이브 마운트 및 경로 설정
drive.mount('/content/drive')
drive_save_path = '/content/drive/MyDrive/my_kobert_absa'

# 2. 커스텀 모델 및 데이터셋 클래스 재정의 (런타임 재시작 대비)
class ABSAKoBERT(nn.Module):
    def __init__(self, bert_model):
        super(ABSAKoBERT, self).__init__()
        self.bert = bert_model
        hidden_size = self.bert.config.hidden_size
        self.confidence_classifier = nn.Linear(hidden_size, 2)
        self.polarity_classifier = nn.Linear(hidden_size, 3)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_token_output = outputs.last_hidden_state[:, 0, :]
        confidence_logits = self.confidence_classifier(cls_token_output)
        polarity_logits = self.polarity_classifier(cls_token_output)
        return confidence_logits, polarity_logits

class ABSADataset(Dataset):
    def __init__(self, texts, aspects, confidences, polarities, tokenizer, max_len=128):
        self.texts = texts
        self.aspects = aspects
        self.confidences = confidences
        self.polarities = polarities
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        aspect = str(self.aspects[item])
        encoding = self.tokenizer(
            text, aspect, add_special_tokens=True, max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'confidence_labels': torch.tensor(self.confidences[item], dtype=torch.long),
            'polarity_labels': torch.tensor(self.polarities[item], dtype=torch.long)
        }

# 3. 디바이스 설정 및 모델/토크나이저 로드
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"현재 사용 기기: {device}")

# 저장된 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(drive_save_path, trust_remote_code=True)
# BERT 베이스 모델 로드 (가중치를 덮어씌우기 위한 뼈대)
bert_base = AutoModel.from_pretrained("skt/kobert-base-v1", trust_remote_code=True)

# 커스텀 모델 구조 초기화
model = ABSAKoBERT(bert_base)
# 드라이브에 저장된 학습 완료 가중치 덮어씌우기
model.load_state_dict(torch.load(f"{drive_save_path}/model_weights2.pth", map_location=device))
model.to(device)
model.eval() # 평가 모드 전환
print("✅ 저장된 모델 가중치 로드 완료!")

# 4. 검증 데이터 준비 및 🚨오류 수정 부분 적용🚨
valid_data = pd.read_csv('Validation_비율통합.csv')
valid_data = valid_data[~valid_data['Aspect'].str.contains('소비전력')]

# ✅ 문제의 원인이었던 라벨 불일치 해결 (+1 적용)
valid_data['SentimentPolarity'] = valid_data['SentimentPolarity'] + 1
valid_data['AspectConfidence'] = 1

test_texts = valid_data['SentimentText'].tolist()
test_aspects = valid_data['Aspect'].tolist()
test_confidence = valid_data['AspectConfidence'].tolist()
test_polarities = valid_data['SentimentPolarity'].tolist()

# 데이터로더 생성
test_dataset = ABSADataset(test_texts, test_aspects, test_confidence, test_polarities, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
print("✅ 테스트 데이터로더 세팅 완료! 평가를 시작합니다...")

# 5. 모델 평가 및 리포트 출력
true_pol, pred_pol = [], []
aspect_for_pol = []

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        _, pol_logits = model(input_ids, attention_mask)
        pol_preds = torch.argmax(pol_logits, dim=1).cpu().numpy()

        for i, conf_val in enumerate(batch['confidence_labels'].numpy()):
            if conf_val == 1:
                pred_pol.append(pol_preds[i])
                true_pol.append(batch['polarity_labels'][i].numpy())

                global_idx = batch_idx * test_loader.batch_size + i
                aspect_for_pol.append(test_aspects[global_idx])

true_pol = np.array(true_pol)
pred_pol = np.array(pred_pol)
aspect_pol_arr = np.array(aspect_for_pol)
unique_aspects = np.unique(np.array(test_aspects))

print("\n============ Aspect 별 감성 극성(Polarity) 요약 리포트 ============\n")

report_data = []
for aspect in unique_aspects:
    idx_pol = (aspect_pol_arr == aspect)
    if np.sum(idx_pol) > 0:
        y_true_sub = true_pol[idx_pol]
        y_pred_sub = pred_pol[idx_pol]

        report_dict = classification_report(
            y_true_sub, y_pred_sub, labels=[0, 1, 2],
            target_names=["부정(-1)", "중립(0)", "긍정(1)"], zero_division=0, output_dict=True
        )
        acc = accuracy_score(y_true_sub, y_pred_sub)

        report_data.append({
            "Aspect": aspect,
            "Accuracy": round(acc, 4),
            "부정 F1": round(report_dict["부정(-1)"]["f1-score"], 4),
            "중립 F1": round(report_dict["중립(0)"]["f1-score"], 4),
            "긍정 F1": round(report_dict["긍정(1)"]["f1-score"], 4),
            "Macro F1": round(report_dict["macro avg"]["f1-score"], 4),
            "Support (데이터 수)": len(y_true_sub)
        })

df_report = pd.DataFrame(report_data)

try:
    display(df_report)
except NameError:
    print(df_report.to_string(index=False))

print("\n" + "=" * 65)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
현재 사용 기기: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/535 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ 저장된 모델 가중치 로드 완료!
✅ 테스트 데이터로더 세팅 완료! 평가를 시작합니다...

============ Aspect 별 감성 극성(Polarity) 요약 리포트 ============



,Aspect,Accuracy,부정 F1,중립 F1,긍정 F1,Macro F1,Support (데이터 수)
0,가격,0.9333,0.7336,0.1728,0.9660,0.6242,2713
1,기능,0.9099,0.7172,0.1707,0.9528,0.6136,2342
2,내구성,0.7968,0.8444,0.0000,0.7755,0.5400,310
3,디자인,0.9209,0.7313,0.3636,0.9555,0.6835,1151
4,무게,0.8520,0.8270,0.0476,0.9036,0.5927,757
5,배터리,0.8432,0.8444,0.0000,0.8670,0.5705,389
6,사이즈,0.8653,0.7063,0.0000,0.9254,0.5439,1359
7,색상,0.8881,0.7222,0.0000,0.9376,0.5533,992
8,소음,0.8190,0.7778,0.2857,0.8700,0.6445,348
9,소재,0.7679,0.7912,0.0000,0.7651,0.5188,168


In [ ]:
# ==========================================
# 🌟 전체 Aspect 통합 감성 극성(Polarity) 리포트 🌟
# ==========================================

print("\n============ 🌟 전체 Aspect 통합 감성 극성 평가 리포트 🌟 ============\n")

# 전체 데이터에 대한 상세 텍스트 리포트 출력
print("[전체 통합 상세 분류 리포트]")
print(classification_report(
    true_pol,
    pred_pol,
    labels=[0, 1, 2],
    target_names=["부정(-1)", "중립(0)", "긍정(1)"],
    zero_division=0
))

# DataFrame으로 요약 표 만들기
overall_report_dict = classification_report(
    true_pol,
    pred_pol,
    labels=[0, 1, 2],
    target_names=["부정(-1)", "중립(0)", "긍정(1)"],
    zero_division=0,
    output_dict=True
)

overall_acc = accuracy_score(true_pol, pred_pol)

overall_data = [{
    "구분": "전체 통합(All Aspects)",
    "Accuracy": round(overall_acc, 4),
    "부정 F1": round(overall_report_dict["부정(-1)"]["f1-score"], 4),
    "중립 F1": round(overall_report_dict["중립(0)"]["f1-score"], 4),
    "긍정 F1": round(overall_report_dict["긍정(1)"]["f1-score"], 4),
    "Macro F1": round(overall_report_dict["macro avg"]["f1-score"], 4),
    "총 데이터 수(Support)": len(true_pol)
}]

df_overall = pd.DataFrame(overall_data)

print("\n[전체 통합 핵심 지표 요약]")
try:
    display(df_overall)
except NameError:
    print(df_overall.to_string(index=False))

print("\n" + "=" * 65)


============ 🌟 전체 Aspect 통합 감성 극성 평가 리포트 🌟 ============

[전체 통합 상세 분류 리포트]
              precision    recall  f1-score   support

      부정(-1)       0.80      0.73      0.76      4333
       중립(0)       0.45      0.12      0.18       615
       긍정(1)       0.91      0.96      0.93     14574

    accuracy                           0.88     19522
   macro avg       0.72      0.60      0.63     19522
weighted avg       0.87      0.88      0.87     19522


[전체 통합 핵심 지표 요약]


,구분,Accuracy,부정 F1,중립 F1,긍정 F1,Macro F1,총 데이터 수(Support)
0,전체 통합(All Aspects),0.8814,0.7637,0.1839,0.9318,0.6265,19522


학습된 ko-ELECTRA와 예상되는 차이점

1. learning_rate=3e-5
2. epochs=3
3. koELECTRA와 ko-BERT의 모델 차이
4. 손실함수 구현방법(compute_loss 부분)..?